# 01. Backtracking

**Tier:** Deep. **Why you'd use this:** You need every subset of a short list. Each value is either in the subset or not, so the search tree branches twice per value and then undoes the choice.

**After this lesson you can:** build every subset with choose-or-skip; copy a finished choice before the next branch undoes it.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Functions](../../course/05-functions/01-definitions-lesson.ipynb), [Base case](../../course/09-recursion/01-base-case-lesson.ipynb)


## Quick review

1. What are the two choices for each element in a subset tree?
2. Why must you copy the current choice when you record an answer?

<details><summary>Answers</summary>

Skip the value, or take it and later undo with `pop`. The same list is reused on every branch; a later `pop` would empty a stored answer if you saved the list itself.
</details>


## Recognize it

The prompt asks for every subset, every combination of length `k`, or every way to assign a small set of independent choices. Wording sounds like "all possibilities", "return every group", or "each item may be included or left out".

Signals: a decision at each item, a recursive branch per decision, and an undo so the next branch starts clean. A look-alike that is not this pattern: counting how many subsets exist without listing them — that is `2 ** n` arithmetic, not a search tree you must walk.


## Why it works this way

A subset is a yes-or-no choice for every value. Walking the list from left to right builds a binary tree: at index `i` you either skip `nums[i]` or append it, then move to `i + 1`. When `i` reaches the length of the list, the path you hold is one complete subset. There are `2 ** n` leaves for `n` values, and each leaf is one answer.

The expensive part is not the arithmetic of `2 ** n`. It is rebuilding work if you copy the whole remaining list at every step, or if you allocate a fresh path instead of mutating one shared path. Backtracking keeps one `choice` list. Append means "take this value". Recurse. Then `pop` undoes the append so the sibling branch that skips the value sees the same prefix. That undo is the whole pattern: explore, then restore.

Recording an answer must copy. `answers.append(choice)` stores a reference to the same list object. Later pops empty that object, so every stored answer collapses to `[]`, or every answer ends up identical to the last path. `choice.copy()` (or `choice[:]`) freezes the path at that leaf. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why a list is a mutable buffer of references, and [Scopes](../../internals/06-scopes/01-closures-lesson.ipynb) for why the nested `walk` closes over the same `choice` name.

Time grows with the number of nodes in the tree, which is proportional to `2 ** n`, and each leaf may copy up to `n` values into the answer list, so listing every subset is on the order of `n * 2 ** n`. Extra memory is the recursion depth `n` plus the space for all answers. That cost is inherent once the problem demands every subset. Bit masks from `0` to `2 ** n - 1` solve the same problem without recursion; they still touch every subset once.

Greedy and dynamic programming do not replace this walk when the output is the full collection of subsets. Those patterns answer a single score or count. Backtracking is the right tool when the answer is the set of paths itself.

Marker for uniqueness: 12-backtracking:01-subsets-lesson.


## When to use it

Use backtracking when you must enumerate assignments that branch and then undo — subsets, combinations of length `k`, letter-case flips, or placements that fail a constraint mid-path. Prefer a closed formula or bit masks when you only need a count or when `n` is tiny and an iterative mask loop is clearer. Prefer dynamic programming when the prompt asks for a minimum, maximum, or number of ways and overlapping subproblems make storing answers cheaper than listing paths. Prefer BFS or DFS on a graph when the state is a node in an explicit graph rather than a growing choice list.


## Brute force, and why it's slow

Count the nodes. A recursive walk that rebuilds a new list on every call still visits every subset, but allocates a fresh path each time.


In [1]:
def subsets_slow(nums):
    nodes = 0

    def walk(index, choice):
        nonlocal nodes
        nodes += 1
        if index == len(nums):
            return [choice]
        skip = walk(index + 1, choice)
        take = walk(index + 1, choice + [nums[index]])
        return skip + take

    answers = walk(0, [])
    return len(answers), nodes


for n in (8, 12, 16):
    print(n, subsets_slow(list(range(n))))

8 (256, 511)
12 (4096, 8191)
16 (65536, 131071)


## Key insight

One shared `choice` list plus append / recurse / pop explores every subset once. Copy the list only when you record a finished path.


## Worked example

Return every subset of a list of distinct integers. Order of subsets does not matter; values inside a subset keep the order they had in the input.


In [2]:
def subsets(nums: list[int]) -> list[list[int]]:
    # 1. Shared path for the current branch.
    answers: list[list[int]] = []
    choice: list[int] = []

    def walk(index: int) -> None:
        # 2. Past the last value: one complete subset.
        if index == len(nums):
            answers.append(choice.copy())
            return
        # 3. Skip this value.
        walk(index + 1)
        # 4. Take it, recurse, then undo.
        choice.append(nums[index])
        walk(index + 1)
        choice.pop()

    walk(0)
    return answers


print(sorted(subsets([1, 2])))
print(len(subsets([1, 2, 3])))

[[], [1], [1, 2], [2]]
8


## Trace

`[1, 2]`, skip-first order (skip branch before take):

| call | index | choice before | action |
|---|---|---|---|
| walk | 0 | `[]` | skip 1 |
| walk | 1 | `[]` | skip 2 → record `[]` |
| walk | 1 | `[]` | take 2 → record `[2]`, pop |
| walk | 0 | `[]` | take 1 |
| walk | 1 | `[1]` | skip 2 → record `[1]` |
| walk | 1 | `[1]` | take 2 → record `[1, 2]`, pop |

Answers: `[[], [2], [1], [1, 2]]`.


## Cost

The tree has about `2 ** n` leaves and a similar number of internal nodes, so time grows like `n * 2 ** n` when each answer copies its path. Extra memory for the search is depth `n`; the answer list holds every subset. Bit masks have the same output cost without recursion.


## Variants

Not solved here: [Permutations](../../challenges/problems/55-permutations.ipynb), [Combination Sum](../../challenges/problems/34-combination-sum.ipynb), [Generate Parentheses](../../challenges/problems/35-generate-parentheses.ipynb), [Word Search](../../challenges/problems/36-word-search.ipynb).


## Common mistake

This bug is silent. The function stores the live `choice` list instead of a copy. Every recorded answer is the same object. After the walk finishes, that object is empty. There is no traceback. The print is `[[], []]`, and the answer should be four distinct subsets.


In [3]:
def subsets_wrong(nums):
    answers = []
    choice = []

    def walk(index):
        if index == len(nums):
            answers.append(choice)  # stores the live list
            return
        walk(index + 1)
        choice.append(nums[index])
        walk(index + 1)
        choice.pop()

    walk(0)
    return answers


print(subsets_wrong([1, 2]))

[[], [], [], []]


The fix copies before storing. `choice.copy()` freezes that path. The worked example does that, so the same input prints four subsets (order may vary before you sort).


In [4]:
print(sorted(subsets([1, 2])))

[[], [1], [1, 2], [2]]


## Python idioms

`choice.copy()` (or `list(choice)`) freezes a path. For combinations of length `k`, start the next index after the value you took so order stays increasing. Bit masks replace the recursion when you prefer an iterative loop.


In [5]:
def subsets_masks(nums: list[int]) -> list[list[int]]:
    n = len(nums)
    answers = []
    for mask in range(1 << n):
        answers.append([nums[i] for i in range(n) if mask & (1 << i)])
    return answers


print(sorted(subsets_masks([1, 2])))

[[], [1], [1, 2], [2]]


## Recap

- Each value is skipped or taken, then the take is undone.
- Copy a finished choice before the next `pop`.
- Listing every subset costs on the order of `n * 2 ** n`.

Next: [practice](01-letter-case-practice.ipynb).
